<!-- codex_annotation: script_overview -->
# 多通道 OME-TIFF 合并

读取 DAPI、488、546、647、cy7 五个已拼接单通道图像，按通道维度堆叠，并写入带通道颜色信息的 OME-TIFF。

注释说明：
- 输入图像尺寸必须一致。
- metadata/OME-XML 用于让 Fiji 按 Composite 多通道图像识别。


In [ ]:
import os
import numpy as np
import tifffile as tiff

input_dir = r"D:\01.analysis\11.test_result\P4-rep2-A-merge\A\P4-rep2-A-stack"
output_file = r"D:\01.analysis\11.test_result\P4-rep2-A-merge\A\P4-rep2-A-stack\merged_channel.ome.tif"

# 1. 读取单通道图像
imgs = [
    tiff.imread(os.path.join(input_dir, "DAPIFused.tif")),  # Channel 0
    tiff.imread(os.path.join(input_dir, "488Fused.tif")),   # Channel 1
    tiff.imread(os.path.join(input_dir, "546Fused.tif")),   # Channel 2
    tiff.imread(os.path.join(input_dir, "647Fused.tif")),   # Channel 3
    tiff.imread(os.path.join(input_dir, "cy7Fused.tif"))    # Channel 4
]

# 堆叠为 (Channels, Height, Width)
merged = np.stack(imgs)
n_channels, height, width = merged.shape

# 2. 构建符合 OME 标准的 XML 元数据
# 关键点：通过 XML 告诉 Fiji 这是一个 Composite 图像，并指定每个通道的颜色
# 颜色使用有符号 32 位整型表示 (ARGB):
# Blue = -16776961, Green = -16711936, Red = -65536, Cyan = -16711681, Magenta = -65281
ome_xml = f"""<?xml version="1.0" encoding="UTF-8"?>
<OME xmlns="http://www.openmicroscopy.org/Schemas/OME/2016-06"
     xmlns:xsi="http://www.w3.org/2000/svg"
     xsi:schemaLocation="http://www.openmicroscopy.org/Schemas/OME/2016-06 http://www.openmicroscopy.org/Schemas/OME/2016-06/ome.xsd">
  <Image ID="Image:0" Name="Merged_5_Channels">
    <Pixels ID="Pixels:0" DimensionOrder="XYCZT" Type="{merged.dtype.name}"
            SizeX="{width}" SizeY="{height}" SizeC="{n_channels}" SizeZ="1" SizeT="1">
      <Channel ID="Channel:0:0" Name="DAPI" Color="-16776961"/>      <Channel ID="Channel:0:1" Name="488" Color="-16711936"/>        <Channel ID="Channel:0:2" Name="546" Color="-65536"/>           <Channel ID="Channel:0:3" Name="647" Color="-16711681"/>        <Channel ID="Channel:0:4" Name="cy7" Color="-65281"/>           </Pixels>
  </Image>
</OME>"""

# 3. 写入带有 OME-XML 的 TIFF 文件
tiff.imwrite(
    output_file,
    merged,
    photometric='minisblack',  # 告诉 Fiji 基础像素值是灰度
    metadata={'axes': 'CYX'},  # 明确轴向
    description=ome_xml.encode('utf-8')  # 注入 OME 元数据
)

print(f"保存完成: {output_file}")
print(f"图像维度: {merged.shape}")